# 1. The problem and the data

**The goal:** flag a company sliding toward financial trouble early, instead of finding out when its yearly accounts are filed months too late.

Before any modelling, I look at the data and the one fact that shapes every choice later: distress is **rare**. Only about 4% of companies in the dataset actually fail, and that rarity is why plain accuracy is a useless score here (a model that calls everyone healthy is 96% accurate and catches nothing).

Data: the Polish Companies Bankruptcy dataset (~10k firms, 64 financial ratios each).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

NAVY, AMBER, GOOD, WATCH, ELEV, BAD = "#0A1628", "#F59E0B", "#22C55E", "#F59E0B", "#F97316", "#EF4444"
plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.25})

## 1. Why accuracy is the wrong metric

The training corpus is the Polish Companies Bankruptcy dataset (UCI id 365): ~10k firms,
64 pre-computed financial ratios, a binary distress label. The label is **rare**.

In [2]:
from src.features import load_horizon, class_balance
df = load_horizon(1)
bal = class_balance(df)
print(f"Companies:            {bal['n']:,}")
print(f"In distress:          {bal['distress']:,}  ({bal['distress_rate']:.1%})")
print(f"Majority-class accuracy: {bal['majority_baseline_accuracy']:.1%}")
print()
print("A model that predicts 'never distress' is",
      f"{bal['majority_baseline_accuracy']:.0%} accurate - and useless.")
print("So we evaluate on PR-AUC (average precision), never accuracy.")

Companies:            7,027
In distress:          271  (3.9%)
Majority-class accuracy: 96.1%

A model that predicts 'never distress' is 96% accurate - and useless.
So we evaluate on PR-AUC (average precision), never accuracy.


## The tool I have to beat: the Altman Z-score

The industry-standard early-warning check is the **Altman Z-score**, a 1968 formula that turns a few numbers from a company's accounts (debt vs assets, profitability, working capital) into a single fragility score. A low score means "looks financially weak."

It is simple and everywhere, but noisy - to catch the real failures you have to investigate a huge pile of healthy companies too. In notebook 2 I compute it and use it as the baseline to beat.